# Channel Performance Analysis

In this notebook I explore, clean, and prepare the raw marketing data for a home decor e-commerce brand. The data covers five channels: Google Ads, Meta Ads, Influencer, Email, and Earned Media - over roughly a year.

My goal here is to get the dataset into a trustworthy, analysis-ready state before I move into Power BI for the modeling, DAX measures, and dashboard building. I run an EDA (exploratory data analysis) to understand the shape and quality of the data, fix inconsistencies I find along the way (channel naming, duplicate rows, missing revenue values), and export a clean CSV at the end.

## 1. Loading the Data

I start by importing pandas and reading the raw CSV export into a dataframe, then take a quick look at the first rows to confirm it loaded correctly.

In [ ]:
# I import pandas, the core library I'll use for all data manipulation in this notebook
import pandas as pd

# I load the raw marketing data CSV into a dataframe
df = pd.read_csv("../data/raw/nordik_home_marketing_data.csv")

# I check the first few rows to make sure the data loaded as expected
df.head()

## 2. Exploratory Data Analysis (EDA)

Before touching anything, I want to understand what I'm working with: how many rows and columns, what data types pandas inferred, and whether there are any obvious data quality issues.

### 2.1 Shape, structure, and summary statistics

In [ ]:
# I check the number of rows and columns in the dataset
df.shape

In [ ]:
# I look at column names, data types, and non-null counts to spot any structural issues early
df.info()

In [ ]:
# I get summary statistics for the numeric columns, to sanity-check ranges and spot outliers
df.describe()

### 2.2 Checking for missing values

In [ ]:
# I count how many rows have a missing 'revenue' value — this is the only column with nulls I've noticed so far
df["revenue"].isnull().sum()

### 2.3 Checking unique values per column

In [ ]:
# I check how many unique values each column has, to get a sense of the data's granularity (e.g. how many distinct channels, campaigns, dates)
df.nunique()

In [ ]:
# I look at the distribution of rows across dates, to confirm the data is spread reasonably evenly over the period
df["date"].value_counts()

In [ ]:
# I check the distinct values in 'channel' — this is where I first spot the naming inconsistency (see next section)
df["channel"].value_counts()

### 2.4 Fixing inconsistent channel naming

When I checked the channel values above, I noticed 'meta ads' (lowercase) sitting alongside 'Meta Ads' — the same channel being counted as two different categories. I standardize this before doing any further analysis, since it would otherwise silently split my Meta Ads numbers in two.

In [ ]:
# I strip any leading/trailing whitespace from channel names, in case that's contributing to inconsistencies
df["channel"] = df["channel"].str.strip()

In [ ]:
# I replace the lowercase 'meta ads' variant with the correctly capitalized 'Meta Ads', so it's no longer treated as a separate category
df["channel"] = df["channel"].replace("meta ads", "Meta Ads")
# alternative approach I considered: df["channel"] = df["channel"].str.title()

In [ ]:
# I re-check the channel values to confirm the fix worked — I should now see only 5 clean categories
df["channel"].value_counts()

## 2 (continued EDA) Exploring Campaigns

In [ ]:
# I check the distinct campaign names and how many rows each has
df["campaign"].value_counts()

In [ ]:
# I group by channel to see which campaigns belong to each channel — this helps me understand the structure of the data
df.groupby("channel").agg(campaign=("campaign","unique"))

In [ ]:
# same check, written a slightly different way, just to confirm the grouping is consistent
df.groupby("channel")["campaign"].unique()

In [ ]:
# I list every unique channel-campaign combination, sorted alphabetically, as a final visual check that everything looks right
df[["channel","campaign"]].drop_duplicates().sort_values(["channel","campaign"],ascending=[True,True])

### 2.5 Checking for duplicate rows

In [ ]:
# I count how many fully duplicated rows exist in the dataset
df.duplicated().sum()

In [ ]:
# I drop the duplicate rows I found, keeping only the first occurrence of each
df = df.drop_duplicates()

In [ ]:
# I confirm there are no duplicates left after the cleanup
df.duplicated().sum()

### 2.6 Handling missing revenue values

Earlier I found 40 rows with a missing 'revenue' value. Instead of dropping them or filling them all with the same value, I want to understand *why* they're missing before deciding how to handle them.

In [ ]:
# I look at the rows with missing revenue, sorted by conversions descending, to see if there's a pattern (e.g. do they have conversions or not?)
df[df["revenue"].isnull()].sort_values(by="conversions",ascending=False)

In [ ]:
# I group by channel and campaign to count, for each group, how many rows there are in total and how many have a null revenue — this helps me see whether the nulls are spread across many campaigns or concentrated in a few
df.groupby(["channel","campaign"]).agg(rows=("revenue","size"), nulls=("revenue", lambda x: x.isnull().sum())).reset_index()

In [ ]:
# Case 1: where revenue is missing AND conversions is 0, it makes sense to fill revenue with 0 — no conversions means no revenue was generated
df.loc[(df["revenue"].isnull()) & (df["conversions"]==0), "revenue"] = 0

In [ ]:
# Case 2: where revenue is missing BUT there were conversions, filling with 0 would understate performance — instead, I estimate the missing value using the average revenue for that same channel-campaign combination
df.loc[
    (df["revenue"].isnull())
    &
    (df["conversions"]!=0),
    "revenue"
] = df.groupby(["channel","campaign"])["revenue"].transform("mean")

In [ ]:
# I confirm there are no missing revenue values left after applying both fixes
df["revenue"].isnull().sum()

## 3. Data Type Conversion

The 'date' column was read in as a generic object/string type. I convert it to a proper datetime type, which I'll need for any time-based analysis later (trends, period-over-period comparisons, etc.) in Power BI.

In [ ]:
# I check the current dtype of the columns, and preview the first 3 rows, before converting the date column
# datatype conversion
df.info()

df.head(3)

In [ ]:
# I convert the 'date' column to a proper datetime type; errors='coerce' means any value that can't be parsed becomes NaT instead of raising an error
df["date"]=pd.to_datetime(df["date"],errors="coerce")

# I check the result to confirm the dtype changed correctly
df.info()
df.head()

In [ ]:
# I double-check that the conversion didn't introduce any new missing dates (which would happen if coerce had to fall back to NaT anywhere)
df["date"].isnull().sum()

## 4. Wrapping Up: Exporting the Clean Dataset

At this point the dataframe is clean: channel names are standardized, duplicates are removed, missing revenue values are handled with clear logic, and the date column has the correct type. I export it to a fresh CSV, which I then load into Power BI to continue the analysis: building the data model, writing the DAX measures, and designing the dashboard.

In [ ]:
# I export the cleaned dataframe to a new CSV file, ready to be loaded into Power BI
df.to_csv("../data/processed/channel_data.csv", index=False)